# **Cross-Family LLM Stitching: Comparing Reconstruction and Prediction-Based Objectives**

*COMS 4705 - Natural Language Processing - Final Project*

### **Authors**

Caroline Zhang (cz2977@columbia.edu)

Siddharth Grover (sg4792@columbia.edu)

Edwin Argudo (eaa2193@columbia.edu)

### **GitHub Repo**

https://github.com/edwin-a14/coms-4705-project-cross-model-stitching


## **Milestone 1**

**Prerequisite:** Run the **entire `smoke-test.ipynb` successfully** before running this notebook. The smoke test notebook will verify setup, model access, activation extraction, output-head reconstruction,
cache reload, and will run a small affine MSE training example. Resolve any failures there first!

### **Deliverables**

| Phase | Deliverable |
|---|---|
| 1. LAMBADA preprocessing and manifests | Versioned, disjoint train/validation/test manifests with aligned labels and preprocessing metadata |
| 2. Sequential activation extraction and persistent caching | Resumable paired Llama/Qwen activation caches, frozen output heads, and provenance |
| 3. Native and constant-activation baselines | Native Llama, native Qwen, and train-mean Qwen baseline metrics with explicit counts |
| 4. Pipeline validation | Reproducible correctness checks, rejection accounting, and an evidence-backed validation report |
| 5. Prep for MSE/CE/KL experiments | Documented cache format, model-free loader, baseline summary, and an untrained affine scaffold |


**Evaluation scope:** predict the first continuation token. Standard full-word LAMBADA evaluation
is a separate protocol. Keep Llama token-level metrics separate from Qwen's because their vocabularies
differ. Prepare the test manifest if needed, but defer test extraction and evaluation until final evaluation.


## **Dependencies & Configuration**

In [1]:
%pip install -q \
    "transformers>=4.45,<5" \
    "accelerate>=1.0" \
    "datasets>=3,<5" \
    "huggingface_hub>=0.34,<1.0" \
    safetensors

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [6]:
import os

from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_DIR = '/content/drive/MyDrive/coms-4705-project'

os.chdir(DRIVE_PROJECT_DIR)
print(f'Working directory set to: {os.getcwd()}')

Mounted at /content/drive
Working directory set to: /content/drive/MyDrive/coms-4705-project


In [8]:
import gc
import random
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
print(f"Running on device: {DEVICE} | Compute dtype: {DTYPE}")

LLAMA_MODEL_ID = "meta-llama/Llama-3.2-3B"
QWEN_MODEL_ID = "Qwen/Qwen2.5-3B"

MAX_PREFIX_TOKENS = 256
TRAIN_SAMPLE_SIZE = 5000   # Target: 5,000 - 10,000 prefixes from training novels
VAL_SAMPLE_SIZE = None     # None = use all 4,869 validation passages (set e.g. 500 for quick smoke test)
TEST_SAMPLE_SIZE = None    # None = use all 5,153 test passages
BATCH_SIZE_EXTRACT = 16    # Adjust based on 24GB L4 VRAM

CACHE_DIR = Path(DRIVE_PROJECT_DIR) / "activations"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

Running on device: cpu | Compute dtype: torch.float16


## **Phase 1 — LAMBADA Preprocessing and Manifests**

In this phase, we'll turn LAMBADA into a consistent set of examples that both models can use.
We'll sample context-and-next-word pairs from the training novels and separate the official
validation and test passages into their context and final word, keeping the original text and
whitespace intact. Each example should use the same text context for Llama and Qwen and fit
within both models' token limits.

Before saving an example, we'll check how each tokenizer handles the boundary between the
context and target word, then identify the correct first continuation token for each model.
We'll save the accepted examples in split-specific manifests: files that record the text,
source IDs, labels, and preprocessing details so extraction can be repeated without resampling.
We'll also track rejected examples and check for overlap across splits. Start with 100 training
and 100 validation examples; scale up after the pilot passes validation. The test manifest can
be prepared here, but test inference stays reserved for final evaluation.


## **Phase 2 — Sequential Activation Extraction and Persistent Caching**

Once the manifests are ready, we'll run each saved context through Llama and Qwen to collect
the representations our stitch will eventually learn to map between. We'll capture the
activation at the last context token, immediately before the final RMSNorm. This gives us
a 3,072-dimensional Llama vector and a 2,048-dimensional Qwen vector for the same text.

We'll load one pretrained model at a time to stay within GPU memory. Activations will be saved to persistent storage in batches,
alongside example IDs, labels, and the information needed to verify or resume extraction.
We'll also save each model's final normalization and output-head weights for baseline evaluation.
Before moving on, we'll confirm that both sets of cached vectors match the same manifest rows
in the same order. Only training and validation examples will be extracted at this stage.

## **Phase 3 — Native and Constant-Activation Baselines**

Before training a stitch, we need reference results to compare it against. We'll evaluate
native Qwen and native Llama using their cached activations and verified output heads, then
add a simple constant baseline that predicts the same Qwen activation for every example.
That constant will be the mean of Qwen's training activations only, so validation data
does not influence it.

For each baseline, we'll report next-token cross-entropy and accuracy, along with accuracy
on the subset of target words represented by a single token. We'll keep training and validation
results separate and include the number of examples behind each score. Llama will use its own
labels and vocabulary, so its token-level results are a separate reference rather than a direct
comparison with Qwen's cross-entropy. These results will give the Milestone 2 experiments a
consistent starting point.


## **Phase 4 — Pipeline Validation**

Here we'll check that the saved data and evaluation code behave as intended before spending
more compute on larger runs. We'll verify that paired vectors belong to the same examples,
labels match the tokenized continuations, splits do not contain unintended overlap, and cached
tensors have the expected shapes and finite values. We'll also check that saved files can be
loaded again after a runtime restart.

On a small set of real examples, we'll compare batched extraction with single-example
extraction and confirm that the saved normalization and output heads reproduce the original
models' logits within an appropriate numerical tolerance. We'll independently check the metric
calculations and confirm that the constant baseline uses training data only. The validation
report should record what was checked, what passed or failed, and what still needs attention;
a check we haven't run remains incomplete.


## **Phase 5 — Prep for MSE/CE/KL experiments**

Finally, we'll organize the validated caches and baseline results so the team can start
Milestone 2 without repeating extraction. We'll document the saved data format and provide
a loader that returns aligned Llama activations, Qwen activations, and Qwen labels without
loading either pretrained 3B model. We'll also prepare the untrained 3,072-to-2,048 affine
mapping that the later experiments will use.

The handoff should explain what each objective needs: paired activations for MSE, Qwen labels
and the frozen Qwen output components for CE, and native Qwen teacher distributions for KL.
Those teacher distributions can be recomputed in batches from cached Qwen activations and
the saved output components. We'll include the baseline summary, configuration details, and
validation report so everyone starts from the same inputs. Training and comparing the three
objectives will happen in the Milestone 2 notebook.
